# Lasso Regression — Hands-on Programming

**Goal.** Implement Lasso from scratch using **coordinate descent** (the textbook Lasso solver from `03_optimization.ipynb` §2). Add a full **$\lambda$-path solver** with warm starts that visits the whole regularisation path in a single pass. Layer **k-fold cross-validation with the 1-SE rule** (`04_statistics.ipynb` §5) on top to pick $\lambda$ from data. Cross-check everything against `sklearn.linear_model.Lasso` and `LassoCV`, and demonstrate variable selection on the diabetes dataset.

**Role of this notebook.** Implementation and empirical validation. Every formula used here was derived in a previous notebook:

| Used here | Where derived |
|---|---|
| Soft-thresholding `S_t(z) = $\text{sign}(z)$ $\cdot$ max(|z|-t, 0)` | `02_mathematics.ipynb` Theorem 4.2 |
| Coordinate-descent update for Lasso | `03_optimization.ipynb` eq. 2.1 |
| `$\lambda_{\max}$ = (2/n) $\cdot$ max_j |⟨$x_j$, y⟩|` (smallest $\lambda$ with $\hat{\theta}$ = 0) | `04_statistics.ipynb` §1.2 |
| 1-SE rule for $\lambda$ selection | `04_statistics.ipynb` eq. 5.1 |

**Prerequisites.** All four prior notebooks in this folder.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → **`05_hands_on_programming`**.

**Plan.**

1. Imports, seeding.
2. `LassoRegression` class — centred features, coordinate descent, soft-thresholding.
3. Cross-check against `sklearn.linear_model.Lasso` on a sparse synthetic problem.
4. Path solver with warm starts; cross-check against `sklearn.linear_model.lasso_path`.
5. k-fold CV + 1-SE rule from scratch; compare with `sklearn.linear_model.LassoCV`.
6. Diabetes — which features does Lasso keep?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random
import time

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LassoCV, lasso_path, LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. `LassoRegression` — coordinate descent from scratch

Three implementation notes:

**(a) Centre, don't penalise the intercept.** Same trick as Ridge: subtract the mean from X (columnwise) and from y; fit the slope on centred data with no bias column; recover the intercept as `ȳ - x̄ $\cdot$ $\hat{\theta}$`. This way the L1 penalty acts only on slopes, never on the intercept.

**(b) sklearn's `$\alpha$` vs our $\lambda$.** Same convention mismatch as Ridge. sklearn's `Lasso($\alpha$=$\alpha$)` minimises `(1/(2n)) $\cdot$ $\|$X$\theta$ - y$\|$^2 + $\alpha$ $\cdot$ $\|\theta\|_1$` (note the factor 1/(2n)). Our convention is `(1/n) $\cdot$ $\|$X$\theta$ - y$\|$^2 + $\lambda$ $\cdot$ $\|\theta\|_1$`. The two differ by a factor of 2 on the penalty: `$\alpha$ = $\lambda$ / 2`. We will pass `$\alpha$ = lam / 2` to sklearn at cross-check time.

**(c) The full residual is maintained incrementally.** Recomputing `r = y - $X \theta$` from scratch is `Θ(n p)` per sweep; updating `r` as each coordinate changes is `Θ(n)` per coordinate, `Θ(n p)` per sweep — same cost asymptotically, but constant-factor cleaner and what every production solver does.

In [ ]:
def soft_thresh(z, t):
    """Soft-thresholding S_t(z) = sign(z) · max(|z| − t, 0). Works on scalar or array z."""
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)


class LassoRegression:
    """Lasso via coordinate descent (Algorithm 2.3 of 03_optimization.ipynb).

    Convention:  L(θ) = (1/n) ‖Xθ − y‖² + lam · ‖θ‖₁
    Centred features → unpenalised intercept.
    """

    def __init__(self, lam=0.1, n_sweeps=500, tol=1e-7):
        self.lam      = lam
        self.n_sweeps = n_sweeps
        self.tol      = tol

    def _coord_descent(self, Xc, yc, theta_init=None):
        n, p = Xc.shape
        c = (Xc * Xc).sum(axis=0)             # c_j = ‖x_j‖²
        theta = np.zeros(p) if theta_init is None else theta_init.copy()
        r = yc - Xc @ theta
        sweep_count = 0
        for sweep_count in range(1, self.n_sweeps + 1):
            max_change = 0.0
            for j in range(p):
                if c[j] == 0:                   # constant column — skip
                    continue
                z_j   = (Xc[:, j] @ r + c[j] * theta[j]) / c[j]
                new   = soft_thresh(z_j, n * self.lam / (2 * c[j]))
                delta = new - theta[j]
                if delta != 0:
                    r -= Xc[:, j] * delta
                    theta[j] = new
                    if abs(delta) > max_change: max_change = abs(delta)
            if max_change < self.tol: break
        return theta, sweep_count

    def fit(self, X, y):
        self.x_mean_ = X.mean(axis=0)
        self.y_mean_ = float(y.mean())
        Xc = X - self.x_mean_
        yc = y - self.y_mean_
        self.coef_, self.n_sweeps_used_ = self._coord_descent(Xc, yc)
        self.intercept_ = self.y_mean_ - self.x_mean_ @ self.coef_
        return self

    def predict(self, X):
        return self.intercept_ + X @ self.coef_

    def active_set(self, tol=1e-8):
        return np.where(np.abs(self.coef_) > tol)[0]


# Smoke test: sparse synthetic.
n, p = 200, 30
X = rng.normal(size=(n, p))
true_theta = np.zeros(p); true_theta[:5] = [3.0, -2.0, 1.5, 1.0, -0.8]
y = X @ true_theta + rng.normal(0, 0.5, size=n)

lr = LassoRegression(lam=0.1).fit(X, y)
print(f"sweeps used    = {lr.n_sweeps_used_}")
print(f"active set     = {lr.active_set()}")
print(f"true non-zero  = {np.where(true_theta != 0)[0]}")
print(f"intercept      = {lr.intercept_:.4f}")

## 2. Cross-check against `sklearn.linear_model.Lasso`

Same data, multiple $\lambda$ values. Translate `lam → $\alpha$ = lam / 2` (see implementation note (b)).

In [ ]:
for lam in [0.01, 0.05, 0.2, 1.0]:
    ours = LassoRegression(lam=lam, tol=1e-10, n_sweeps=10000).fit(X, y)
    skl  = Lasso(alpha=lam / 2, tol=1e-10, max_iter=100000).fit(X, y)
    diff = np.max(np.abs(ours.coef_ - skl.coef_))
    n_ours = int(np.sum(np.abs(ours.coef_) > 1e-8))
    n_skl  = int(np.sum(np.abs(skl.coef_) > 1e-8))
    print(f"λ = {lam:>5g}  →  max |coef diff| = {diff:.2e}    # active (ours / skl) = {n_ours} / {n_skl}")

**Reading.** Coefficients agree to ~1e-6 or better; active sets agree exactly. Tiny numerical differences come from the slightly different convergence tolerances in the two implementations.

## 3. $\lambda$-path solver with warm starts

Three steps, as outlined in `03_optimization.ipynb` §5:

1. Compute `$\lambda_{\max}$ = (2/n) $\cdot$ max_j |⟨$x_j$, y⟩|` on the centred data.
2. Build a log-spaced grid from $\lambda_{\max}$ down to `eps $\cdot$ $\lambda_{\max}$` (typical `eps = 1e-3`).
3. Walk the grid from large to small $\lambda$. At each step initialise coordinate descent from the *previous* solution (warm start) — usually only a few sweeps are needed.

In [ ]:
def lasso_path_ours(X, y, n_lams=80, eps=1e-3, tol=1e-7, n_sweeps=10000):
    """Compute the Lasso path from λ_max down to eps·λ_max with warm starts."""
    x_mean = X.mean(axis=0); y_mean = y.mean()
    Xc = X - x_mean; yc = y - y_mean
    n, p = Xc.shape
    lam_max = 2.0 / n * np.max(np.abs(Xc.T @ yc))
    lams = np.geomspace(lam_max, lam_max * eps, n_lams)
    coefs = np.empty((n_lams, p))
    theta = np.zeros(p)
    cd = LassoRegression(tol=tol, n_sweeps=n_sweeps)
    for i, lam in enumerate(lams):
        cd.lam = lam
        theta, _ = cd._coord_descent(Xc, yc, theta_init=theta)   # warm start
        coefs[i] = theta
    return lams, coefs

t0 = time.perf_counter()
lams_ours, coefs_ours = lasso_path_ours(X, y, n_lams=80)
ours_t = time.perf_counter() - t0

# sklearn path. Note alphas = lams / 2 in our convention.
t0 = time.perf_counter()
alphas_skl, coefs_skl, _ = lasso_path(X - X.mean(axis=0), y - y.mean(),
                                       alphas=lams_ours / 2, tol=1e-7)
skl_t = time.perf_counter() - t0
coefs_skl = coefs_skl.T  # shape (n_alphas, p)

print(f"ours  path  ({len(lams_ours)} λ):  {ours_t * 1000:.1f} ms")
print(f"sklearn path ({len(lams_ours)} λ):  {skl_t * 1000:.1f} ms")
print(f"max |coefs ours − sklearn| over the path: {np.max(np.abs(coefs_ours - coefs_skl)):.2e}")

In [ ]:
# Visualise our Lasso path on this synthetic problem.
fig, ax = plt.subplots(figsize=(8, 4.5))
for j in range(p):
    is_true = j < 5
    ax.plot(lams_ours, coefs_ours[:, j],
            color=("crimson" if is_true else "lightgray"),
            lw=(1.5 if is_true else 0.8))
ax.set_xscale("log")
ax.set_xlabel("λ (log)")
ax.set_ylabel("coefficient")
ax.axhline(0, color="black", lw=0.5)
ax.set_title("Lasso path (our solver)  —  red = true non-zero features, grey = noise")
plt.show()

**Reading.** The red curves (true features) come alive earliest as $\lambda$ decreases from $\lambda_{\max}$. The grey curves (noise features) stay flat at zero across a wide range of $\lambda$, then leak in near the OLS end. The piecewise-linear segments are clearly visible — kinks correspond to features entering or leaving the active set.

## 4. CV + 1-SE rule from scratch

Algorithm 5.2 of `04_statistics.ipynb`, implemented end-to-end. We then cross-check the picked $\lambda$ against `sklearn.linear_model.LassoCV`.

In [ ]:
def lasso_cv_1se(X, y, k=5, n_lams=80, eps=1e-3, seed=0):
    """k-fold CV over a path of λ values, return both λ_min and λ_1SE."""
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    fold_err = []
    lams_path = None
    for tr, te in kf.split(X):
        lams, coefs = lasso_path_ours(X[tr], y[tr], n_lams=n_lams, eps=eps)
        x_tr_mean = X[tr].mean(axis=0); y_tr_mean = y[tr].mean()
        # Intercept for each λ: y_tr_mean − x_tr_mean · θ̂(λ)
        intercepts = y_tr_mean - coefs @ x_tr_mean
        preds = X[te] @ coefs.T + intercepts                  # (n_te, n_lams)
        err = np.mean((y[te][:, None] - preds) ** 2, axis=0)
        fold_err.append(err)
        lams_path = lams                                       # paths from each fold use same λs
    fold_err = np.array(fold_err)                              # (k, n_lams)
    cv = fold_err.mean(axis=0)
    se = fold_err.std(axis=0) / np.sqrt(k)

    i_min  = int(np.argmin(cv))
    thresh = cv[i_min] + se[i_min]
    # Largest λ with CV ≤ threshold. Path is decreasing in λ from index 0.
    i_1se  = int(np.min(np.where(cv <= thresh)[0]))
    return {
        "lams":  lams_path,
        "cv":    cv,
        "se":    se,
        "lam_min": lams_path[i_min],
        "lam_1se": lams_path[i_1se],
    }

res = lasso_cv_1se(X, y, k=5, seed=SEED)

# sklearn baseline (note alpha = lam / 2)
skl_cv = LassoCV(alphas=res["lams"] / 2, cv=5, random_state=SEED, fit_intercept=True).fit(X, y)
lam_skl = skl_cv.alpha_ * 2

print(f"ours: λ_min  = {res['lam_min']:.4f}")
print(f"ours: λ_1SE  = {res['lam_1se']:.4f}")
print(f"sklearn:     = {lam_skl:.4f}  (LassoCV picks λ_min)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.errorbar(res["lams"], res["cv"], yerr=res["se"],
            fmt="o-", color="crimson", lw=1, markersize=3)
ax.axvline(res["lam_min"], color="black",   ls="--", label=f"λ_min  = {res['lam_min']:.3f}")
ax.axvline(res["lam_1se"], color="seagreen", ls="--", label=f"λ_1SE  = {res['lam_1se']:.3f}")
ax.set_xscale("log")
ax.set_xlabel("λ (log)")
ax.set_ylabel("5-fold CV MSE ± SE")
ax.set_title("Our CV path with the 1-SE rule")
ax.legend()
plt.show()

n_act_min = int(np.sum(np.abs(LassoRegression(lam=res['lam_min']).fit(X, y).coef_) > 1e-8))
n_act_1se = int(np.sum(np.abs(LassoRegression(lam=res['lam_1se']).fit(X, y).coef_) > 1e-8))
print(f"true non-zero       = {int(np.sum(true_theta != 0))}")
print(f"at λ_min : selected = {n_act_min}")
print(f"at λ_1SE : selected = {n_act_1se}")

**Reading.** The 1-SE rule picks a noticeably sparser model than `$\lambda_{\min}$`. On this problem `$\lambda_{1SE}$` usually matches the true support of size 5 most closely — at the cost of a slightly larger CV error. That trade is exactly what the 1-SE rule is designed for.

## 5. Diabetes — which features does Lasso keep?

Same diabetes regression dataset as the previous folders. We fit OLS, Ridge (`$\lambda$` from `RidgeCV`), and Lasso (`$\lambda_{\min}$` and `$\lambda_{1SE}$`); compare the test RMSE and look at which features each selects.

In [ ]:
data = load_diabetes()
X_full, y_full, feat_names = data.data, data.target, data.feature_names
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=SEED)

# Our Lasso CV on the training set.
diab_cv = lasso_cv_1se(X_tr, y_tr, k=5, n_lams=80, eps=1e-4, seed=SEED)
lasso_min = LassoRegression(lam=diab_cv["lam_min"]).fit(X_tr, y_tr)
lasso_1se = LassoRegression(lam=diab_cv["lam_1se"]).fit(X_tr, y_tr)
ols       = LinearRegression().fit(X_tr, y_tr)

def report(name, est):
    y_pred_te = est.predict(X_te)
    coef = est.coef_
    n_act = int(np.sum(np.abs(coef) > 1e-8))
    return {
        "name":      name,
        "rmse_te":   float(np.sqrt(mean_squared_error(y_te, y_pred_te))),
        "r2_te":     float(r2_score(y_te, y_pred_te)),
        "n_active":  n_act,
    }

rows = [
    report("OLS",                                       ols),
    report(f"Lasso (λ_min  = {diab_cv['lam_min']:.4g})", lasso_min),
    report(f"Lasso (λ_1SE  = {diab_cv['lam_1se']:.4g})", lasso_1se),
]
print(f"{'method':<32}  {'RMSE_test':>11}  {'R²_test':>8}  {'# features':>11}")
for r in rows:
    print(f"{r['name']:<32}  {r['rmse_te']:>11.2f}  {r['r2_te']:>8.4f}  {r['n_active']:>11}")

print("\nFeatures selected by λ_min:")
print("  ", [n for n, c in zip(feat_names, lasso_min.coef_) if abs(c) > 1e-8])
print("\nFeatures selected by λ_1SE:")
print("  ", [n for n, c in zip(feat_names, lasso_1se.coef_) if abs(c) > 1e-8])

In [ ]:
# Visualise the Lasso path on diabetes; mark the CV-selected λ values.
lams_d, coefs_d = lasso_path_ours(X_tr, y_tr, n_lams=80, eps=1e-4)
fig, ax = plt.subplots(figsize=(8, 4.5))
for j, name in enumerate(feat_names):
    ax.plot(lams_d, coefs_d[:, j], lw=1.4, label=name)
ax.set_xscale("log")
ax.axvline(diab_cv["lam_min"], color="black",   ls="--", label=f"λ_min  = {diab_cv['lam_min']:.3f}")
ax.axvline(diab_cv["lam_1se"], color="seagreen", ls="--", label=f"λ_1SE  = {diab_cv['lam_1se']:.3f}")
ax.set_xlabel("λ (log)")
ax.set_ylabel("coefficient value")
ax.set_title("Lasso regularisation path on diabetes")
ax.axhline(0, color="black", lw=0.5)
ax.legend(fontsize=7, ncol=2, loc="upper right")
plt.show()

**Reading.** On diabetes (10 features, ~440 patients), Lasso achieves roughly the same test RMSE as OLS while using *fewer* features. The 1-SE rule typically drops 3–4 features and keeps the strongest 6–7 (bmi, bp, the lipid-related s* group, sex, age). For an explanatory model this is preferable; for raw prediction the difference is small.

## Takeaway

- **Coordinate descent in ~40 lines.** Each step is a soft-thresholding (Theorem 4.2 of `02_mathematics.ipynb`); the full residual is maintained incrementally. Converges in 10–50 sweeps on typical problems.
- **Cross-check.** Our coefficients match `sklearn.linear_model.Lasso` to ~1e-6, exact active sets, modulo the convention switch `$\alpha$ = lam / 2`.
- **Path solver with warm starts.** Visits 80 $\lambda$ values in a few milliseconds; cross-checks exactly with `sklearn.linear_model.lasso_path`.
- **CV + 1-SE rule.** Picks a sparser, more stable model than the raw CV minimum. Implementation is one short function on top of the path solver.
- **Diabetes.** Lasso achieves OLS-comparable RMSE while keeping a sparser, more interpretable feature set. The 1-SE rule is the natural default when interpretation matters.

Next: `../05_logistic_regression/01_intuition.ipynb`.

**This concludes the Lasso Regression folder.** Up next in module 01 is `05_logistic_regression/` — same regularised-loss recipe (with L1 / L2 / Elastic Net penalties), but with a *logistic* loss for binary classification instead of squared error for regression.